# Single Data Pull For SEC EDGAR API Tag

In [1]:
import pandas as pd
import numpy as np
import requests
from edgar_functions import get_cik, get_submissions, get_company_facts
from headers import headers

## Description of Program

This program allows for the user to input a publicly traded companies ticker and the desired data to be extracted from the SEC EDGAR API and it will get saved to a DataFrame.

Enter Relevant Data Below:

In [2]:
ticker = 'CHTR'

In [3]:
cik = get_cik(ticker, headers=headers)
print(cik)

0001091667


## Use CIK to Populate Submissions and Company Facts Data

In [4]:
submissions = get_submissions(cik, headers=headers)
print(submissions.keys())

dict_keys(['cik', 'entityType', 'sic', 'sicDescription', 'ownerOrg', 'insiderTransactionForOwnerExists', 'insiderTransactionForIssuerExists', 'name', 'tickers', 'exchanges', 'ein', 'lei', 'description', 'website', 'investorWebsite', 'category', 'fiscalYearEnd', 'stateOfIncorporation', 'stateOfIncorporationDescription', 'addresses', 'phone', 'flags', 'formerNames', 'filings'])


In [5]:
company_facts = get_company_facts(cik, headers=headers)
print(company_facts.keys())

dict_keys(['cik', 'entityName', 'facts'])


## Print Company Information and Available 10-K Data

In [ ]:
# Company name and fiscal year end
print(submissions['name'])
print("Fiscal Year End: ", submissions['fiscalYearEnd'])

CHARTER COMMUNICATIONS, INC. /MO/
Fiscal Year End:  1231


In [7]:
# Add list of filings to DataFrame and show 10-Ks available
df_filings = pd.DataFrame(submissions["filings"]["recent"])
df_filings.loc[df_filings["form"] == "10-K"]

,accessionNumber,filingDate,reportDate,acceptanceDateTime,act,form,fileNumber,filmNumber,items,core_type,size,isXBRL,isInlineXBRL,primaryDocument,primaryDocDescription
61,0001091667-25-000034,2025-01-31,2024-12-31,2025-01-31T12:02:28.000Z,34,10-K,001-33664,25575895,,XBRL,14865912,1,1,chtr-20241231.htm,10-K
181,0001091667-24-000028,2024-02-02,2023-12-31,2024-02-02T12:03:03.000Z,34,10-K,001-33664,24588784,,XBRL,18578220,1,1,chtr-20231231.htm,10-K
313,0001091667-23-000024,2023-01-27,2022-12-31,2023-01-27T12:04:46.000Z,34,10-K,001-33664,23559617,,XBRL,14753302,1,1,chtr-20221231.htm,10-K
417,0001091667-22-000024,2022-01-28,2021-12-31,2022-01-28T12:05:57.000Z,34,10-K,001-33664,22565296,,XBRL,16207673,1,1,chtr-20211231.htm,10-K
549,0001091667-21-000022,2021-01-29,2020-12-31,2021-01-29T12:06:30.000Z,34,10-K,001-33664,21568073,,XBRL,16249953,1,1,chtr-20201231.htm,10-K
690,0001091667-20-000024,2020-01-31,2019-12-31,2020-01-31T12:08:17.000Z,34,10-K,001-33664,20563292,,XBRL,25833994,1,1,chtr12312019-10k.htm,10-K
794,0001091667-19-000029,2019-01-31,2018-12-31,2019-01-31T12:13:55.000Z,34,10-K,001-33664,19553667,,10-K,24438359,1,0,chtr12312018-10k.htm,10-K
873,0001091667-18-000025,2018-02-02,2017-12-31,2018-02-02T13:07:24.000Z,34,10-K,001-33664,18568722,,10-K,24655722,1,0,chtr12312017-10k.htm,10-K
942,0001091667-17-000030,2017-02-16,2016-12-31,2017-02-16T12:10:06.000Z,34,10-K,001-33664,17616441,,10-K,25719126,1,0,chtr123116-10k.htm,10-K


## Extract XBRL Financial Data

In [8]:
# Function used to extract data from SEC EDGAR XBRL Company Facts Dictionary
def extract_cy_sec_data(company_facts_dict, account):
    df = pd.DataFrame(company_facts_dict["facts"]["us-gaap"][account]["units"]["USD"])
    df = df.loc[
        (df["form"] == "10-K") & 
        (df["frame"].str.contains("^CY\d{4}$", regex=True))]
    return df

In [9]:
# Function used to clean the DataFrame exported from XBRL Company Facts
def clean_xbrl_df_data(df_to_clean, name_of_account):
    # Create copy of df
    df = df_to_clean.copy()
    # Add ticker column
    df["ticker"] = ticker
    # Add year column
    df["year"] = df["end"].str.slice(0, 4)
    # Select columns to keep "val", "ticker", "year"
    df = df.loc[:, ["val", "ticker", "year"]]
    # Rename val column
    df = df.rename(columns={"val": str(name_of_account)})
    # Reset the index
    df = df.set_index(["ticker", "year"])

    return df

In [11]:
# Create DataFrames
df_revenues = extract_cy_sec_data(company_facts, "Revenues")
df_operating_expenses = extract_cy_sec_data(company_facts, "CostsAndExpenses")
df_operating_income = extract_cy_sec_data(company_facts, "OperatingIncomeLoss")
df_depreciation_amortization = extract_cy_sec_data(company_facts, "DepreciationAmortizationAndAccretionNet")
df_net_income = extract_cy_sec_data(company_facts, "NetIncomeLoss")

In [12]:
df_revenues

,start,end,val,accn,fy,fp,form,filed,frame
7,2010-01-01,2010-12-31,7059000000,0001091667-13-000020,2012,FY,10-K,2013-02-22,CY2010
24,2011-01-01,2011-12-31,7204000000,0001091667-14-000080,2013,FY,10-K,2014-02-21,CY2011
44,2012-01-01,2012-12-31,7504000000,0001091667-15-000049,2014,FY,10-K,2015-02-24,CY2012
65,2013-01-01,2013-12-31,8155000000,0001091667-16-000396,2015,FY,10-K,2016-02-10,CY2013
86,2014-01-01,2014-12-31,9108000000,0001091667-17-000030,2016,FY,10-K,2017-02-16,CY2014
107,2015-01-01,2015-12-31,9754000000,0001091667-18-000025,2017,Q4,10-K,2018-02-02,CY2015
128,2016-01-01,2016-12-31,29003000000,0001091667-19-000029,2018,Q4,10-K,2019-01-31,CY2016
149,2017-01-01,2017-12-31,41581000000,0001091667-20-000024,2019,FY,10-K,2020-01-31,CY2017
170,2018-01-01,2018-12-31,43634000000,0001091667-21-000022,2020,FY,10-K,2021-01-29,CY2018
191,2019-01-01,2019-12-31,45764000000,0001091667-22-000024,2021,FY,10-K,2022-01-28,CY2019


In [13]:
# Clean DataFrames
df_revenues = clean_xbrl_df_data(df_revenues, "Revenues")
df_operating_expenses = clean_xbrl_df_data(df_operating_expenses, "OperatingExpenses")
df_operating_income = clean_xbrl_df_data(df_operating_income, "OperatingIncome")
df_depreciation_amortization = clean_xbrl_df_data(df_depreciation_amortization, "DepreciationAndAmortization")
df_net_income = clean_xbrl_df_data(df_net_income, "NetIncome")

In [14]:
df_revenues

Revenues
ticker year             
CHTR   2010   7059000000
       2011   7204000000
       2012   7504000000
       2013   8155000000
       2014   9108000000
       2015   9754000000
       2016  29003000000
       2017  41581000000
       2018  43634000000
       2019  45764000000
       2020  48097000000
       2021  51682000000
       2022  54022000000
       2023  54607000000
       2024  55085000000

In [15]:
# Combine the DataFrames into once source
df_income_data = pd.merge(df_revenues, df_operating_expenses, left_index=True, right_index=True)
df_income_data = pd.merge(df_income_data, df_operating_income, left_index=True, right_index=True)
df_income_data = pd.merge(df_income_data, df_depreciation_amortization, left_index=True, right_index=True)
df_income_data = pd.merge(df_income_data, df_net_income, left_index=True, right_index=True)
df_income_data

Revenues  OperatingExpenses  OperatingIncome  \
ticker year                                                    
CHTR   2010   7059000000         6035000000       1024000000   
       2011   7204000000         6163000000       1041000000   
       2012   7504000000         6589000000        915000000   
       2013   8155000000         7246000000        909000000   
       2014   9108000000         8137000000        971000000   
       2015   9754000000         8640000000       1114000000   
       2016  29003000000        26547000000       2456000000   
       2017  41581000000        37475000000       4106000000   
       2018  43634000000        38413000000       5221000000   
       2019  45764000000        39253000000       6511000000   
       2020  48097000000        39692000000       8405000000   
       2021  51682000000        41156000000      10526000000   
       2022  54022000000        42060000000      11962000000   
       2023  54607000000        42048000000      12559000000   
       2024  55085000000        41967000000      13118000000   

             DepreciationAndAmortization   NetIncome  
ticker year                                           
CHTR   2010                   1524000000  -237000000  
       2011                   1592000000  -369000000  
       2012                   1713000000  -304000000  
       2013                   1854000000  -169000000  
       2014                   2102000000  -183000000  
       2015                   2125000000  -271000000  
       2016                   6907000000  3522000000  
       2017                  10588000000  9895000000  
       2018                  10318000000  1230000000  
       2019                   9926000000  1668000000  
       2020                   9704000000  3222000000  
       2021                   9345000000  4654000000  
       2022                   8903000000  5055000000  
       2023                   8696000000  4557000000  
       2024                   8673000000  5083000000

In [16]:
# Add additional income information
df_income_data["EBITDA"] = df_income_data["OperatingIncome"] + df_income_data["DepreciationAndAmortization"]
df_income_data["OperatingMargin"] = df_income_data["OperatingIncome"] / df_income_data["Revenues"]
df_income_data["NetIncomeMargin"] = df_income_data["NetIncome"] / df_income_data["Revenues"]
df_income_data["EBITDAMargin"] = df_income_data["EBITDA"] / df_income_data["Revenues"]
df_income_data["RevenueGrowth"] = df_income_data["Revenues"].pct_change()
df_income_data["EBITDAGrowth"] = df_income_data["EBITDA"].pct_change()

In [17]:
df_income_data

Revenues  OperatingExpenses  OperatingIncome  \
ticker year                                                    
CHTR   2010   7059000000         6035000000       1024000000   
       2011   7204000000         6163000000       1041000000   
       2012   7504000000         6589000000        915000000   
       2013   8155000000         7246000000        909000000   
       2014   9108000000         8137000000        971000000   
       2015   9754000000         8640000000       1114000000   
       2016  29003000000        26547000000       2456000000   
       2017  41581000000        37475000000       4106000000   
       2018  43634000000        38413000000       5221000000   
       2019  45764000000        39253000000       6511000000   
       2020  48097000000        39692000000       8405000000   
       2021  51682000000        41156000000      10526000000   
       2022  54022000000        42060000000      11962000000   
       2023  54607000000        42048000000      12559000000   
       2024  55085000000        41967000000      13118000000   

             DepreciationAndAmortization   NetIncome       EBITDA  \
ticker year                                                         
CHTR   2010                   1524000000  -237000000   2548000000   
       2011                   1592000000  -369000000   2633000000   
       2012                   1713000000  -304000000   2628000000   
       2013                   1854000000  -169000000   2763000000   
       2014                   2102000000  -183000000   3073000000   
       2015                   2125000000  -271000000   3239000000   
       2016                   6907000000  3522000000   9363000000   
       2017                  10588000000  9895000000  14694000000   
       2018                  10318000000  1230000000  15539000000   
       2019                   9926000000  1668000000  16437000000   
       2020                   9704000000  3222000000  18109000000   
       2021                   9345000000  4654000000  19871000000   
       2022                   8903000000  5055000000  20865000000   
       2023                   8696000000  4557000000  21255000000   
       2024                   8673000000  5083000000  21791000000   

             OperatingMargin  NetIncomeMargin  EBITDAMargin  RevenueGrowth  \
ticker year                                                                  
CHTR   2010         0.145063        -0.033574      0.360958            NaN   
       2011         0.144503        -0.051222      0.365491       0.020541   
       2012         0.121935        -0.040512      0.350213       0.041644   
       2013         0.111465        -0.020723      0.338811       0.086754   
       2014         0.106610        -0.020092      0.337396       0.116861   
       2015         0.114210        -0.027783      0.332069       0.070927   
       2016         0.084681         0.121436      0.322829       1.973447   
       2017         0.098747         0.237969      0.353383       0.433679   
       2018         0.119654         0.028189      0.356121       0.049374   
       2019         0.142273         0.036448      0.359169       0.048815   
       2020         0.174751         0.066990      0.376510       0.050979   
       2021         0.203669         0.090051      0.384486       0.074537   
       2022         0.221428         0.093573      0.386232       0.045277   
       2023         0.229989         0.083451      0.389236       0.010829   
       2024         0.238141         0.092276      0.395589       0.008753   

             EBITDAGrowth  
ticker year                
CHTR   2010           NaN  
       2011      0.033359  
       2012     -0.001899  
       2013      0.051370  
       2014      0.112197  
       2015      0.054019  
       2016      1.890707  
       2017      0.569369  
       2018      0.057506  
       2019      0.057790  
       2020      0.101722  
       2021      0.097300  
       2022      0.050023  
      

In [ ]:
df_income_data.to_csv('chtr-income-data.csv', index=True)